# Schema profiler — metadata only

Builds a profile of the data in the attached lakehouse **without exporting any real values**.
The profile is used outside this tenant to generate fake data with the same structure and behaviour.

**How to run**
1. Attach the lakehouse you want to profile as this notebook's default lakehouse.
2. Run all.
3. Download `Files/profiling/output/schema-profile-<timestamp>.json` and hand it over.

No settings are needed. By default it profiles every table in the default lakehouse and every
folder under `Files/`. To narrow the scope or add Warehouse tables, put an optional
`Files/profiling/profile_request.json` in place (see the README).

**Safety:** only aggregate queries are run, nothing is displayed, category labels are masked,
text is reduced to shapes, and a final check blocks the output file if anything looks like a real value.

In [ ]:
import datetime as dt
import gzip
import json
import math
import os
import re
import unicodedata
from collections import Counter, defaultdict

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T

PROFILER_VERSION = "1.0.0"
spark = globals().get("spark") or SparkSession.builder.getOrCreate()

DEFAULTS = {
    "tables": ["*"],                    # "*" = every table in the default lakehouse
    "auto_discover_files": True,        # profile every folder under Files/
    "file_datasets": [],                # [{"path": "landing/sales", "format": "csv", "options": {...}}]
    "exclude_file_folders": [],
    "warehouse_tables": [],             # ["MyWarehouse.dbo.Orders"]
    "allowed_real_label_columns": [],   # ["orders.status"] — owner approved only
    "source_form_overrides": {},        # {"orders": "sql_database"}
    "foreign_keys": [],                 # [{"from": "orders.customer_id", "to": "customers.customer_id"}]
    "business_keys": {},                # {"orders": ["order_no", "line_no"]}
    "sample_above_rows": 50_000_000,
    "sample_target_rows": 20_000_000,
    "categorical_max_distinct": 100,
    "max_rule_columns": 12,
}
OUTPUT_DIR = "profiling"
PCTS = [0.0, 0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99, 1.0]
PKEYS = ["p0", "p1", "p5", "p25", "p50", "p75", "p95", "p99", "p100"]
PLACEHOLDER_STRINGS = ["N/A", "NA", "n/a", "na", "NULL", "null", "None", "none", "-", "--", "?",
                       "UNKNOWN", "Unknown", "unknown", "TBD", "#N/A"]
PLACEHOLDER_NUMBERS = [-1, -99, -999, 999, 9999, 99999]
PLACEHOLDER_DATES = ["1900-01-01", "1899-12-30", "1970-01-01", "2099-12-31", "9999-12-31"]
FILE_FORMATS = {"csv": "csv", "tsv": "csv", "parquet": "parquet", "json": "json", "jsonl": "json",
                "ndjson": "json", "avro": "avro", "orc": "orc"}
COMPRESSIONS = {"gz": "gzip", "bz2": "bzip2", "snappy": "snappy", "zst": "zstd", "lz4": "lz4"}
GUID_RE = r"^[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12}$"
HASH_RE = r"^[0-9a-fA-F]{32,128}$"
SOFT_DELETE_RE = re.compile(r"(^|_)(is_?)?(deleted|removed|inactive|active)($|_)|deleted_(at|on|date|ts)", re.I)
WATERMARK_RE = re.compile(r"modif|updat|chang|load|ingest|insert|creat|etl|last", re.I)
TIME_COL_RE = re.compile(r"creat|insert|event|order|transaction|txn|booking|date|_dt$|_ts$|time", re.I)

# --- where am I running ------------------------------------------------------------------------
FABRIC_MOUNT = "/lakehouse/default"
IN_FABRIC = os.path.isdir(os.path.join(FABRIC_MOUNT, "Files"))
LAKEHOUSE_ROOT = FABRIC_MOUNT if IN_FABRIC else os.environ.get("PROFILER_LAKEHOUSE_ROOT", "")
if not LAKEHOUSE_ROOT or not os.path.isdir(os.path.join(LAKEHOUSE_ROOT, "Files")):
    raise RuntimeError("No default lakehouse found. Attach the lakehouse to profile as this "
                       "notebook's default lakehouse, then run again.")


def os_path(*rel):
    return os.path.join(LAKEHOUSE_ROOT, "Files", *rel)


def spark_files_path(rel):
    return f"Files/{rel}" if IN_FABRIC else os_path(rel)


# --- small helpers -----------------------------------------------------------------------------
def q(name):
    return ".".join("`" + p.replace("`", "``") + "`" for p in name.split("."))


def col(name):
    return F.col("`" + name.replace("`", "``") + "`")


def num(x):
    if x is None:
        return None
    try:
        x = float(x)
    except (TypeError, ValueError):
        return None
    return None if math.isnan(x) or math.isinf(x) else x


def sig3(x):
    x = num(x)
    if x is None:
        return None
    if x == 0:
        return 0
    v = round(x, -int(math.floor(math.log10(abs(x)))) + 2)
    return int(v) if v == int(v) and abs(v) < 1e15 else v


def pct(n, d):
    return round(100.0 * n / d, 1) if d else None


def safe_err(e):
    # Never keep the message: Spark messages can quote data values.
    return type(e).__name__


def py_shape(s, maxlen=60):
    out = []
    for ch in str(s)[:maxlen]:
        cat = unicodedata.category(ch)
        if cat == "Lu":
            out.append("A")
        elif cat.startswith("L"):
            out.append("a")
        elif cat.startswith("N"):
            out.append("9")
        elif cat.startswith("M"):
            continue
        else:
            out.append(ch)
    return "".join(out)


def spark_shape(c, maxlen=60):
    s = F.substring(c.cast("string"), 1, maxlen)
    s = F.regexp_replace(s, r"\p{M}", "")
    s = F.regexp_replace(s, r"\p{Lu}", "A")
    s = F.regexp_replace(s, r"[\p{Ll}\p{Lt}\p{Lm}\p{Lo}]", "a")
    return F.regexp_replace(s, r"\p{N}", "9")


def value_like_name(name):
    """True when a schema field name looks like data (an ID, date or number) rather than a name."""
    return bool(re.fullmatch(r"[\d\-_.:/ ]+", name) or re.fullmatch(GUID_RE, name)
                or re.fullmatch(HASH_RE, name))


TS_TYPES = (T.TimestampType,) + ((T.TimestampNTZType,) if hasattr(T, "TimestampNTZType") else ())


def family(dtype):
    if isinstance(dtype, T.BooleanType):
        return "boolean"
    if isinstance(dtype, T.IntegralType):
        return "integer"
    if isinstance(dtype, T.NumericType):
        return "number"
    if isinstance(dtype, T.DateType):
        return "date"
    if isinstance(dtype, TS_TYPES):
        return "timestamp"
    if isinstance(dtype, T.StringType):
        return "string"
    if isinstance(dtype, T.BinaryType):
        return "binary"
    if isinstance(dtype, (T.StructType, T.ArrayType, T.MapType)):
        return "complex"
    return "other"


def schema_tree(dtype):
    """Nested field names and types, with value-like names masked."""
    if isinstance(dtype, T.StructType):
        return {(py_shape(f.name) if value_like_name(f.name) else f.name): schema_tree(f.dataType)
                for f in dtype.fields}
    if isinstance(dtype, T.ArrayType):
        return {"array_of": schema_tree(dtype.elementType)}
    if isinstance(dtype, T.MapType):
        return {"map_key": schema_tree(dtype.keyType), "map_value": schema_tree(dtype.valueType)}
    return dtype.simpleString()


def load_request():
    req = dict(DEFAULTS)
    p = os_path(OUTPUT_DIR, "profile_request.json")
    if os.path.isfile(p):
        with open(p, encoding="utf-8") as fh:
            user = json.load(fh)
        unknown = set(user) - set(DEFAULTS)
        if unknown:
            raise ValueError(f"Unknown keys in profile_request.json: {sorted(unknown)}")
        req.update(user)
    req["_allowed"] = {s.lower() for s in req["allowed_real_label_columns"]}
    return req

In [ ]:
# --- one dataset (table, file folder or warehouse table) ---------------------------------------
def classify(fam, st, n_nonnull, cat_max):
    d = st.get("distinct") or 0
    near_unique = n_nonnull > 0 and d >= 0.98 * n_nonnull
    if fam == "boolean":
        return "boolean"
    if fam == "integer":
        if near_unique and st["nulls"] == 0 and n_nonnull > 20:
            return "id"
        return "categorical" if d <= 20 else "numeric"
    if fam == "number":
        return "numeric"
    if fam in ("date", "timestamp"):
        return fam
    if fam == "string":
        if d <= cat_max and (d <= 20 or d <= 0.5 * n_nonnull):
            return "categorical"
        if (st.get("avg_len") or 0) > 40 and pct(st.get("spaces", 0), n_nonnull or 1) > 50:
            return "free_text"
        return "id" if near_unique and n_nonnull > 20 else "text_code"
    if fam == "binary":
        return "binary"
    if fam == "complex":
        return "json"
    return "other"


def profile_dataframe(name, df, form, req, extra=None):
    fields = list(df.schema.fields)
    row_count = df.count()
    method = {"type": "full_scan"}
    sdf = df
    if row_count > req["sample_above_rows"]:
        frac = req["sample_target_rows"] / row_count
        sdf = df.sample(False, frac, seed=42)
        method = {"type": "sample", "sample_pct": round(100 * frac, 2)}
    sdf = sdf.cache()
    raw = {"categories": {}, "cond_null": []}
    try:
        table = _profile(name, sdf, fields, row_count, form, req, raw)
    finally:
        sdf.unpersist()
    table["profiling_method"] = method
    if extra:
        table["source"].update(extra.get("source", {}))
        for k, v in extra.items():
            if k != "source":
                table[k] = v
    return table, raw


def _profile(name, sdf, fields, row_count, form, req, raw):
    cat_max = req["categorical_max_distinct"]
    fams = [family(f.dataType) for f in fields]

    # pass 1: nulls, distinct, string basics
    exprs = [F.count(F.lit(1)).alias("n")]
    for i, (f, fam) in enumerate(zip(fields, fams)):
        c = col(f.name)
        exprs.append(F.sum(F.when(c.isNull(), 1).otherwise(0)).alias(f"nul_{i}"))
        if fam not in ("complex", "binary", "other"):
            exprs.append(F.approx_count_distinct(c, 0.02).alias(f"dst_{i}"))
        if fam == "string":
            exprs += [F.sum(F.when(F.trim(c) == "", 1).otherwise(0)).alias(f"emp_{i}"),
                      F.avg(F.length(c)).alias(f"avl_{i}"),
                      F.sum(F.when(F.instr(F.trim(c), " ") > 0, 1).otherwise(0)).alias(f"spc_{i}")]
    r = sdf.agg(*exprs).collect()[0].asDict()
    n = r["n"] or 0

    stats = []
    for i, (f, fam) in enumerate(zip(fields, fams)):
        st = {"nulls": r.get(f"nul_{i}") or 0, "distinct": r.get(f"dst_{i}"),
              "empty": r.get(f"emp_{i}"), "avg_len": r.get(f"avl_{i}"), "spaces": r.get(f"spc_{i}") or 0}
        st["kind"] = classify(fam, st, n - st["nulls"], cat_max) if n else "unknown"
        stats.append(st)

    # exact uniqueness for id candidates
    for f, st in zip(fields, stats):
        if st["kind"] == "id":
            nn = n - st["nulls"]
            exact = sdf.select(col(f.name)).where(col(f.name).isNotNull()).distinct().count()
            st["distinct"], st["unique"] = exact, exact == nn

    cols_out = []
    for f, fam, st in zip(fields, fams, stats):
        nn = n - st["nulls"]
        c_out = {
            "name": f.name,
            "source_type": f.dataType.simpleString() if form != "files" else f"text in file; Spark read as {f.dataType.simpleString()}",
            "spark_type": f.dataType.simpleString(),
            "nullable_in_schema": f.nullable,
            "null_pct": pct(st["nulls"], n),
            "kind": st["kind"],
            "pii": None,
        }
        if fam == "string":
            c_out["empty_string_pct"] = pct(st["empty"] or 0, n)
        if st["distinct"] is not None:
            c_out["distinct_count"] = sig3(st["distinct"])
            c_out["distinct_pct"] = pct(st["distinct"], nn)
            c_out["distinct_method"] = "exact" if st["kind"] == "id" else "approx_2pct"
            c_out["unique"] = bool(st.get("unique", False))
        if fam == "complex":
            c_out["json"] = {"schema_tree": schema_tree(f.dataType)}
        cols_out.append(c_out)

    if n:
        _numeric_pass(sdf, fields, fams, stats, cols_out, n)
        _temporal_pass(sdf, fields, fams, stats, cols_out, n)
        _categorical_pass(name, sdf, fields, stats, cols_out, n, raw)
        _string_pass(sdf, fields, fams, stats, cols_out, n)
        _placeholder_pass(sdf, fields, fams, cols_out, n)

    pk = [c["name"] for c, st in zip(cols_out, stats) if st.get("unique") and st["nulls"] == 0]
    table = {
        "name": name,
        "description": None,
        "source": {"form": req["source_form_overrides"].get(name, form)},
        "row_count": sig3(row_count),
        "primary_key_candidates": pk,
        "primary_key": pk[:1],
        "business_key": req["business_keys"].get(name, []),
        "columns": cols_out,
    }
    if n:
        table["duplicates"] = _duplicates(sdf, fields, n, req["business_keys"].get(name))
        table["growth"] = _growth(cols_out, stats)
        table["change_behaviour"] = {
            "soft_delete_column_candidates": [f.name for f in fields if SOFT_DELETE_RE.search(f.name)],
            "watermark_column_candidates": [f.name for f, fam in zip(fields, fams)
                                            if fam in ("date", "timestamp") and WATERMARK_RE.search(f.name)],
        }
        table["rules"] = _date_order_rules(sdf, fields, fams, req)
        table["correlations"] = _correlations(sdf, fields, stats, req)
        _cond_null_rules(sdf, fields, stats, cols_out, n, req, raw)
    return table


def _numeric_pass(sdf, fields, fams, stats, cols_out, n):
    idx = [i for i, fam in enumerate(fams) if fam in ("integer", "number")]
    if not idx:
        return
    exprs = []
    for i in idx:
        c = col(fields[i].name)
        exprs += [F.percentile_approx(c, PCTS, 10000).alias(f"pq_{i}"), F.avg(c).alias(f"av_{i}"),
                  F.stddev(c).alias(f"sd_{i}"), F.sum(F.when(c == 0, 1).otherwise(0)).alias(f"z_{i}"),
                  F.sum(F.when(c < 0, 1).otherwise(0)).alias(f"ng_{i}"),
                  F.min(c).alias(f"mn_{i}"), F.max(c).alias(f"mx_{i}")]
        if fams[i] == "number" and not isinstance(fields[i].dataType, T.DecimalType):
            txt = c.cast("string")
            exprs.append(F.max(F.when(txt.rlike(r"^-?\d+\.\d+$"), F.length(F.substring_index(txt, ".", -1)))
                               .otherwise(0)).alias(f"dc_{i}"))
    r = sdf.agg(*exprs).collect()[0].asDict()
    for i in idx:
        st, out, dtype = stats[i], cols_out[i], fields[i].dataType
        if st["kind"] == "id":
            lo, hi = num(r[f"mn_{i}"]), num(r[f"mx_{i}"])
            span = (hi - lo + 1) if lo is not None and hi is not None else None
            out["id_pattern"] = {
                "type": "integer_sequence",
                "digits": len(str(abs(int(hi)))) if hi is not None else None,
                "gaps_pct": pct(span - st["distinct"], span) if span else None,
            }
            continue
        qs = r[f"pq_{i}"] or [None] * len(PCTS)
        block = {k: sig3(v) for k, v in zip(PKEYS, qs)}
        block.update({
            "mean": sig3(r[f"av_{i}"]), "stddev": sig3(r[f"sd_{i}"]),
            "zero_pct": pct(r[f"z_{i}"] or 0, n), "negative_pct": pct(r[f"ng_{i}"] or 0, n),
            "decimals_used": dtype.scale if isinstance(dtype, T.DecimalType) else (0 if fams[i] == "integer" else r.get(f"dc_{i}")),
            "shape": _shape_of_distribution(qs, r[f"av_{i}"]),
        })
        out["numeric"] = block


def _shape_of_distribution(qs, mean):
    vals = [num(v) for v in qs]
    if any(v is None for v in vals) or vals[8] == vals[0]:
        return "constant" if vals and vals[0] is not None and vals[8] == vals[0] else None
    p5, p50, p95 = vals[2], vals[4], vals[6]
    if p95 == p5:
        return "concentrated"
    skew = ((p95 - p50) - (p50 - p5)) / (p95 - p5)
    return "skewed_right" if skew > 0.3 else "skewed_left" if skew < -0.3 else "roughly_symmetric"


def _temporal_pass(sdf, fields, fams, stats, cols_out, n):
    for i, fam in enumerate(fams):
        if fam not in ("date", "timestamp"):
            continue
        c = col(fields[i].name)
        aggs = [F.min(F.to_date(c)).alias("mn"), F.max(F.to_date(c)).alias("mx")]
        if fam == "timestamp":
            aggs.append(F.sum(F.when(F.date_format(c, "HH:mm:ss.SSS") != "00:00:00.000", 1).otherwise(0)).alias("tp"))
        r = sdf.agg(*aggs).collect()[0].asDict()
        nn = n - stats[i]["nulls"]
        block = {
            "min_day": r["mn"].isoformat() if r["mn"] else None,
            "max_day": r["mx"].isoformat() if r["mx"] else None,
            "has_time_part": bool(r.get("tp")) if fam == "timestamp" else False,
            "timezone": f"unknown (Spark session time zone: {spark.conf.get('spark.sql.session.timeZone')})",
        }
        months = sdf.where(c.isNotNull()).groupBy(F.date_format(c, "yyyy-MM").alias("m")).count().collect()
        if len(months) > 240:
            years = Counter()
            for m in months:
                years[m["m"][:4]] += m["count"]
            block["rows_by_year_pct"] = {y: pct(v, nn) for y, v in sorted(years.items())}
        else:
            block["rows_by_month_pct"] = {m["m"]: pct(m["count"], nn) for m in sorted(months, key=lambda x: x["m"])}
        stats[i]["_months"] = {m["m"]: m["count"] for m in months}
        keys = [F.dayofweek(c).alias("wd")] + ([F.hour(c).alias("hr")] if fam == "timestamp" else [])
        wh = sdf.where(c.isNotNull()).groupBy(*keys).count().collect()
        wd, hr = Counter(), Counter()
        for row in wh:
            wd[row["wd"]] += row["count"]
            if fam == "timestamp":
                hr[row["hr"]] += row["count"]
        names = {1: "sun", 2: "mon", 3: "tue", 4: "wed", 5: "thu", 6: "fri", 7: "sat"}
        block["weekday_pct"] = {names[k]: pct(v, nn) for k, v in sorted(wd.items())}
        if fam == "timestamp" and block["has_time_part"]:
            block["hour_pct"] = {f"{k:02d}": pct(v, nn) for k, v in sorted(hr.items())}
        cols_out[i]["temporal"] = block


def _categorical_pass(name, sdf, fields, stats, cols_out, n, raw):
    for i, st in enumerate(stats):
        if st["kind"] not in ("categorical", "boolean"):
            continue
        c = col(fields[i].name)
        rows = (sdf.where(c.isNotNull()).groupBy(c.alias("v")).count()
                .orderBy(F.desc("count")).limit(1000).collect())
        nn = n - st["nulls"]
        if st["kind"] == "boolean":
            cols_out[i]["categorical"] = {"labels": "boolean",
                                          "values": [{"label": str(r["v"]).lower(), "pct": pct(r["count"], nn)} for r in rows]}
        else:
            # Real values stay in memory only; labels are assigned after all datasets are read.
            raw["categories"][fields[i].name] = [(r["v"], r["count"]) for r in rows]
            cols_out[i]["categorical"] = {"_pending": True, "_nonnull": nn}


def _string_pass(sdf, fields, fams, stats, cols_out, n):
    for i, fam in enumerate(fams):
        kind = stats[i]["kind"]
        if fam != "string" or kind not in ("id", "text_code", "free_text"):
            continue
        c = col(fields[i].name)
        nn = n - stats[i]["nulls"]
        r = sdf.where(c.isNotNull()).agg(
            F.percentile_approx(F.length(c), [0.0, 0.5, 0.95, 1.0], 10000).alias("ln"),
            F.sum(F.when(c.rlike(GUID_RE), 1).otherwise(0)).alias("guid"),
            F.sum(F.when(c.rlike(HASH_RE), 1).otherwise(0)).alias("hash"),
        ).collect()[0].asDict()
        ln = r["ln"] or [None] * 4
        length = {"min": ln[0], "p50": ln[1], "p95": ln[2], "max": ln[3]}
        if kind == "free_text":
            cols_out[i]["free_text"] = {"length": length, "language": None}
            continue
        shapes = (sdf.where(c.isNotNull()).groupBy(spark_shape(c).alias("s")).count()
                  .orderBy(F.desc("count")).limit(10).collect())
        top = [{"shape": s["s"], "pct": pct(s["count"], nn)} for s in shapes]
        if kind == "id":
            g, h = pct(r["guid"] or 0, nn) or 0, pct(r["hash"] or 0, nn) or 0
            cols_out[i]["id_pattern"] = {
                "type": "guid" if g >= 95 else "hash" if h >= 95 else "natural_code",
                "length": length, "top_shapes": top,
            }
        else:
            cols_out[i]["text_shape"] = {"length": length, "top_shapes": top}


def _placeholder_pass(sdf, fields, fams, cols_out, n):
    exprs, keys = [], []
    for i, fam in enumerate(fams):
        c = col(fields[i].name)
        is_id = cols_out[i]["kind"] == "id"
        if fam == "string" and not is_id:
            cands, test = PLACEHOLDER_STRINGS, lambda v: F.trim(c) == v
        elif fam in ("integer", "number"):
            cands = [v for v in PLACEHOLDER_NUMBERS if v < 0 or not is_id]
            test = lambda v: c == v
        elif fam in ("date", "timestamp"):
            cands, test = PLACEHOLDER_DATES, lambda v: F.to_date(c) == F.to_date(F.lit(v))
        else:
            continue
        for j, v in enumerate(cands):
            exprs.append(F.sum(F.when(test(v), 1).otherwise(0)).alias(f"ph_{i}_{j}"))
            keys.append((i, j, v))
    if not exprs:
        return
    r = sdf.agg(*exprs).collect()[0].asDict()
    found = defaultdict(list)
    for i, j, v in keys:
        cnt = r[f"ph_{i}_{j}"] or 0
        # Large positive numbers are only placeholders when they stand out.
        floor = 0.005 if isinstance(v, int) and v > 0 else 0.001
        if cnt and cnt >= floor * n:
            found[i].append({"value": v, "pct": pct(cnt, n)})
    for i, items in found.items():
        cols_out[i]["placeholders"] = items


def _duplicates(sdf, fields, n, business_key):
    out = {}
    hashable = [col(f.name) for f in fields if not isinstance(f.dataType, T.MapType)]
    if hashable:
        d = sdf.select(F.xxhash64(*hashable).alias("h")).agg(F.countDistinct("h")).collect()[0][0]
        out["exact_row_pct"] = pct(n - d, n)
    if business_key:
        d = sdf.select(*[col(k) for k in business_key]).distinct().count()
        out["business_key_dup_pct"] = pct(n - d, n)
    return out


def _growth(cols_out, stats):
    cands = [(c, st) for c, st in zip(cols_out, stats) if st.get("_months")]
    if not cands:
        return {"time_column": None}
    cands.sort(key=lambda x: (0 if TIME_COL_RE.search(x[0]["name"]) else 1, x[0]["null_pct"] or 0))
    c, st = cands[0]
    this_year = dt.date.today().year
    months = sorted((m, v) for m, v in st["_months"].items() if m and 1950 <= int(m[:4]) <= this_year + 1)
    trend = "unknown"
    months = months[1:-1]  # first and last months are often partial
    if len(months) >= 6:
        third = len(months) // 3
        first = sum(v for _, v in months[:third]) / third
        last = sum(v for _, v in months[-third:]) / third
        trend = "growing" if last > 1.2 * first else "shrinking" if last < 0.8 * first else "steady"
    return {"time_column": c["name"], "trend": trend, "months_covered": len(st["_months"]),
            "notes": "rows by month: see this column's temporal block"}


def _date_order_rules(sdf, fields, fams, req):
    tcols = [f.name for f, fam in zip(fields, fams) if fam in ("date", "timestamp")][: req["max_rule_columns"]]
    pairs = [(a, b) for k, a in enumerate(tcols) for b in tcols[k + 1:]]
    if not pairs:
        return []
    exprs = []
    for k, (a, b) in enumerate(pairs):
        ca, cb = col(a).cast("timestamp"), col(b).cast("timestamp")
        both = ca.isNotNull() & cb.isNotNull()
        exprs += [F.sum(F.when(both, 1).otherwise(0)).alias(f"b_{k}"),
                  F.sum(F.when(both & (ca <= cb), 1).otherwise(0)).alias(f"le_{k}"),
                  F.sum(F.when(both & (ca >= cb), 1).otherwise(0)).alias(f"ge_{k}")]
    r = sdf.agg(*exprs).collect()[0].asDict()
    rules = []
    for k, (a, b) in enumerate(pairs):
        both = r[f"b_{k}"] or 0
        if both < 30:
            continue
        le, ge = pct(r[f"le_{k}"] or 0, both), pct(r[f"ge_{k}"] or 0, both)
        if le >= 95 and ge < 95:
            rules.append(f"{a} <= {b} in {le}% of rows where both are set")
        elif ge >= 95 and le < 95:
            rules.append(f"{b} <= {a} in {ge}% of rows where both are set")
        elif le >= 95 and ge >= 95:
            rules.append(f"{a} = {b} (same day/time) in {min(le, ge)}% of rows where both are set")
    return rules


def _correlations(sdf, fields, stats, req):
    ncols = [f.name for f, st in zip(fields, stats) if st["kind"] == "numeric"][: req["max_rule_columns"]]
    pairs = [(a, b) for k, a in enumerate(ncols) for b in ncols[k + 1:]]
    if not pairs:
        return []
    r = sdf.agg(*[F.corr(col(a).cast("double"), col(b).cast("double")).alias(f"c_{k}")
                  for k, (a, b) in enumerate(pairs)]).collect()[0].asDict()
    out = []
    for k, (a, b) in enumerate(pairs):
        v = num(r[f"c_{k}"])
        if v is not None and abs(v) >= 0.7:
            out.append({"columns": [a, b], "description": f"linear correlation r = {round(v, 2)}"})
    return out


def _cond_null_rules(sdf, fields, stats, cols_out, n, req, raw):
    cats = [f.name for f, st in zip(fields, stats)
            if st["kind"] == "categorical" and 2 <= (st["distinct"] or 0) <= 20][:5]
    targets = [c["name"] for c in cols_out if c["null_pct"] is not None and 5 <= c["null_pct"] <= 95][:20]
    for cat in cats:
        tg = [t for t in targets if t != cat]
        if not tg:
            continue
        rows = sdf.groupBy(col(cat).alias("v")).agg(
            F.count(F.lit(1)).alias("n"),
            *[F.sum(F.when(col(t).isNull(), 1).otherwise(0)).alias(f"t_{k}") for k, t in enumerate(tg)]
        ).collect()
        for row in rows:
            if row["v"] is None or row["n"] < max(30, 0.01 * n):
                continue
            for k, t in enumerate(tg):
                rate = pct(row[f"t_{k}"] or 0, row["n"])
                overall = next(c["null_pct"] for c in cols_out if c["name"] == t)
                if (rate >= 99 or rate <= 1) and abs(rate - overall) >= 30:
                    raw["cond_null"].append((cat, row["v"], t, rate, overall))
    raw["cond_null"] = raw["cond_null"][:30]

In [ ]:
# --- discovery and readers ---------------------------------------------------------------------
def lakehouse_tables(req):
    if req["tables"] != ["*"]:
        return list(req["tables"])
    out = []
    for t in spark.catalog.listTables():
        if t.isTemporary or (t.tableType or "").upper() == "VIEW":
            continue
        out.append(t.name)
    return sorted(out)


def delta_metadata(name):
    src, change = {}, {}
    try:
        d = spark.sql(f"DESCRIBE DETAIL {q(name)}").collect()[0].asDict()
        src.update({"table_format": d.get("format"), "partition_columns": list(d.get("partitionColumns") or []),
                    "num_files": sig3(d.get("numFiles")), "size_bytes": sig3(d.get("sizeInBytes"))})
    except Exception as e:  # noqa: BLE001
        src["detail_error"] = safe_err(e)
    try:
        # Only the operation name, the write mode and numeric metrics are read.
        # Never predicates or user names: they can contain values.
        h = (spark.sql(f"DESCRIBE HISTORY {q(name)} LIMIT 500")
             .select("timestamp", "operation", F.col("operationParameters")["mode"].alias("mode"), "operationMetrics")
             .collect())
        ops, modes, times, written = Counter(), Counter(), [], []
        merge = Counter()
        for row in h:
            ops[row["operation"]] += 1
            if row["mode"]:
                modes[row["mode"]] += 1
            times.append(row["timestamp"])
            m = row["operationMetrics"] or {}
            if row["operation"] in ("WRITE", "CREATE TABLE AS SELECT", "REPLACE TABLE AS SELECT",
                                    "CREATE OR REPLACE TABLE AS SELECT", "STREAMING UPDATE"):
                if m.get("numOutputRows"):
                    written.append(float(m["numOutputRows"]))
            if row["operation"] == "MERGE":
                for k in ("numTargetRowsInserted", "numTargetRowsUpdated", "numTargetRowsDeleted"):
                    merge[k] += float(m.get(k) or 0)
        times.sort()
        gaps = [(b - a).total_seconds() / 3600 for a, b in zip(times, times[1:])]
        gaps.sort()
        change.update({
            "history_commits_seen": len(h),
            "operations": dict(ops),
            "write_modes": dict(modes),
            "median_hours_between_commits": sig3(gaps[len(gaps) // 2]) if gaps else None,
            "rows_per_write_p50": sig3(sorted(written)[len(written) // 2]) if written else None,
            "hard_deletes": "yes" if ops.get("DELETE") or merge["numTargetRowsDeleted"] else "none_seen",
        })
        if merge:
            ins, upd = merge["numTargetRowsInserted"], merge["numTargetRowsUpdated"]
            change["merge_updated_vs_inserted_pct"] = pct(upd, ins + upd)
        if ops.get("MERGE", 0) >= max(ops.get("WRITE", 0), 1):
            change["load_pattern_guess"] = "incremental (merge)"
        elif modes.get("Overwrite", 0) > modes.get("Append", 0):
            change["load_pattern_guess"] = "full (overwrite)"
        elif modes.get("Append"):
            change["load_pattern_guess"] = "incremental (append)"
        else:
            change["load_pattern_guess"] = "unknown"
        change["history_note"] = "Delta history may be trimmed by log retention."
    except Exception as e:  # noqa: BLE001
        change["history_error"] = safe_err(e)
    return src, change


def file_folders(req):
    folders = [d["path"] for d in req["file_datasets"]]
    if req["auto_discover_files"]:
        skip = {OUTPUT_DIR, *req["exclude_file_folders"], *folders}
        for e in sorted(os.listdir(os_path())):
            if e not in skip and not e.startswith((".", "_")) and os.path.isdir(os_path(e)):
                folders.append(e)
    return folders


def inspect_folder(rel):
    base = os_path(rel)
    if os.path.isdir(os.path.join(base, "_delta_log")):
        return {"format": "delta", "files": [], "partitions": []}
    files, parts = [], set()
    for dirpath, dirnames, filenames in os.walk(base):
        dirnames[:] = [d for d in dirnames if not d.startswith((".", "_"))]
        for seg in os.path.relpath(dirpath, base).split(os.sep):
            if "=" in seg:
                parts.add(seg.split("=", 1)[0])
        for fn in filenames:
            if not fn.startswith((".", "_")) and not fn.endswith(".crc"):
                files.append(os.path.join(dirpath, fn))
        if len(files) >= 200_000:
            break
    exts, comps = Counter(), Counter()
    for p in files:
        bits = os.path.basename(p).lower().split(".")
        comp = COMPRESSIONS.get(bits[-1]) if len(bits) > 1 else None
        if comp:
            bits = bits[:-1]
        exts[FILE_FORMATS.get(bits[-1]) if len(bits) > 1 else None] += 1
        comps[comp or "none"] += 1
    fmt = exts.most_common(1)[0][0] if exts else None
    return {"format": fmt, "mixed": len(exts) > 1, "files": files, "partitions": sorted(parts),
            "compression": comps.most_common(1)[0][0] if comps else None}


def _first_lines(path, k=2, nbytes=65536):
    opener = gzip.open if path.lower().endswith(".gz") else open
    with opener(path, "rb") as fh:
        chunk = fh.read(nbytes)
    if chunk.startswith(b"\xef\xbb\xbf"):
        enc = "utf-8-sig"
    elif chunk.startswith((b"\xff\xfe", b"\xfe\xff")):
        enc = "utf-16"
    else:
        try:
            chunk[:-4].decode("utf-8")
            enc = "utf-8"
        except UnicodeDecodeError:
            enc = "not utf-8 (unknown)"
    text = chunk.decode("utf-8", errors="replace").lstrip("﻿")
    return enc, text.splitlines()[:k]


def _looks_typed(v):
    v = v.strip().strip('"')
    return bool(re.fullmatch(r"-?\d+([.,]\d+)?", v) or re.fullmatch(r"\d{4}-\d{2}-\d{2}.*", v)
                or re.fullmatch(r"\d{1,2}/\d{1,2}/\d{2,4}.*", v))


def csv_header_check(lines, sep):
    """Header only when line 1 looks like names and line 2 has typed values where line 1 does not.
    Without that contrast we cannot be sure line 1 is not data, so names get masked."""
    if len(lines) < 2:
        return False
    h, d = lines[0].split(sep), lines[1].split(sep)
    names_ok = (len(set(h)) == len(h) and
                all(re.fullmatch(r"[A-Za-z_][\w \-.()/#%]{0,63}", x.strip().strip('"')) for x in h))
    contrast = any(_looks_typed(b) and not _looks_typed(a) for a, b in zip(h, d))
    return names_ok and contrast


def read_file_dataset(rel, spec, info):
    fmt = spec.get("format") or info["format"]
    opts = dict(spec.get("options", {}))
    extra = {"source": {"file": {"format": fmt, "compression": info.get("compression")}}}
    header = True
    if fmt == "csv":
        enc, lines = _first_lines(info["files"][0])
        sep = opts.get("sep") or max([",", ";", "|", "\t"], key=lambda s: lines[0].count(s) if lines else 0)
        if "header" in opts:
            header = str(opts["header"]).lower() == "true"
        else:
            # Unconfirmed header: read line 1 as data, so a data row can never become column names.
            header = csv_header_check(lines, sep)
        opts = {"sep": sep, "header": str(header).lower(), "inferSchema": "true", **opts}
        extra["source"]["file"].update({"delimiter": sep, "header": header, "encoding": enc,
                                        "header_note": None if header or "header" in spec.get("options", {})
                                        else "header not confirmed; names masked. Set options.header in profile_request.json"})
    elif fmt == "json" and "multiLine" not in opts:
        _, lines = _first_lines(info["files"][0], k=1)
        opts["multiLine"] = "true" if lines and lines[0].lstrip().startswith("[") else "false"
    reader = spark.read.format(fmt).options(**opts)
    if not info["partitions"]:
        reader = reader.option("recursiveFileLookup", "true")
    df = reader.load(spark_files_path(rel))
    if not header:
        df = df.toDF(*[f"col_{k + 1:02d}" for k in range(len(df.columns))])
    if fmt == "json":
        df = df.drop("_corrupt_record") if "_corrupt_record" in df.columns else df
    names = Counter(py_shape(os.path.basename(p)) for p in info["files"])
    total = sum(names.values())
    per_file = df.groupBy(F.input_file_name().alias("f")).count().agg(
        F.percentile_approx("count", [0.5], 1000).alias("p50")).collect()[0]["p50"]
    extra["source"]["file"].update({
        "file_count": total,
        "file_name_shapes": [{"shape": s, "pct": pct(c, total)} for s, c in names.most_common(10)],
        "rows_per_file_p50": sig3(per_file[0]) if per_file else None,
    })
    extra["source"]["partition_columns"] = info["partitions"]
    return df, extra


def read_warehouse_table(name):
    import com.microsoft.spark.fabric  # noqa: F401  (Fabric Spark connector for Warehouse)
    return spark.read.synapsesql(name)

In [ ]:
# --- masking, foreign keys, safety check, output -----------------------------------------------
def assign_labels(results, req):
    """Mask category labels. Columns with the same name in several datasets share one mapping."""
    by_domain = defaultdict(list)
    for t, raw in results:
        for cname in raw["categories"]:
            by_domain[cname.lower()].append((t, cname))
    for dom, members in by_domain.items():
        union = Counter()
        for t, cname in members:
            raw = next(r for tt, r in results if tt is t)
            for v, cnt in raw["categories"][cname]:
                union[str(v)] += cnt
        order = {v: f"v{k + 1:02d}" for k, (v, _) in enumerate(union.most_common())}
        for t, cname in members:
            raw = next(r for tt, r in results if tt is t)
            column = next(c for c in t["columns"] if c["name"] == cname)
            nn = column["categorical"].pop("_nonnull")
            column["categorical"].pop("_pending")
            real = f"{t['name']}.{cname}".lower() in req["_allowed"]
            column["categorical"].update({
                "labels": "real" if real else "masked",
                "values": [{"label": (str(v) if real else order[str(v)]), "pct": pct(cnt, nn)}
                           for v, cnt in raw["categories"][cname]],
                "same_domain_as": [f"{o['name']}.{oc}" for o, oc in members if o is not t] or None,
            })
            raw["_labels"] = raw.get("_labels", {})
            raw["_labels"][cname] = (lambda v, real=real, order=order: str(v) if real else order[str(v)])
    for t, raw in results:
        for cat, v, target, rate, overall in raw["cond_null"]:
            label = raw["_labels"][cat](v)
            t["rules"].append(f"{target} is null in {rate}% of rows where {cat} = {label} (overall {overall}%)")


def infer_foreign_keys(results, loaders, req):
    pk_cols = {}
    for t, _ in results:
        for c in t.get("primary_key_candidates", []):
            fam = next(col_["spark_type"] for col_ in t["columns"] if col_["name"] == c)
            pk_cols[(t["name"], c.lower())] = (c, fam)
    cands = []
    for spec in req["foreign_keys"]:
        (ct, cc), (pt, pc) = spec["from"].rsplit(".", 1), spec["to"].rsplit(".", 1)
        cands.append((ct, cc, pt, pc, "declared"))
    for t, _ in results:
        for c in t["columns"]:
            for (pt, pcl), (pc, ptype) in pk_cols.items():
                if pt == t["name"]:
                    continue
                stem = pt.split(".")[-1].split(":")[-1].split("/")[-1].lower()
                cl = c["name"].lower()
                name_match = (cl == pcl != "id") or (pcl == "id" and cl in (f"{stem}_id", f"{stem.rstrip('s')}_id"))
                if name_match and family_of(c["spark_type"]) == family_of(ptype):
                    cands.append((t["name"], c["name"], pt, pc, "inferred_by_name"))
    seen, by_name = set(), {t["name"]: t for t, _ in results}
    for ct, cc, pt, pc, how in cands:
        if (ct, cc) in seen or ct not in loaders or pt not in loaders:
            continue
        seen.add((ct, cc))
        try:
            child = loaders[ct]().select(col(cc).alias("k")).where(F.col("k").isNotNull())
            parent = loaders[pt]().select(col(pc).alias("k")).distinct()
            child_n = child.count()
            if not child_n:
                continue
            matched = child.join(parent, "k", "left_semi").count()
            per = child.groupBy("k").count()
            cpp = per.agg(F.percentile_approx("count", [0.5, 0.95, 1.0], 1000).alias("q")).collect()[0]["q"]
            parents_n, used = parent.count(), per.join(parent, "k", "left_semi").count()
            fk = {"columns": [cc], "references": f"{pt}({pc})", "found_by": how,
                  "match_pct": pct(matched, child_n), "orphan_pct": pct(child_n - matched, child_n),
                  "children_per_parent": {"p50": cpp[0], "p95": cpp[1], "max": sig3(cpp[2]),
                                          "parents_with_zero_pct": pct(parents_n - used, parents_n)}}
            if how == "declared" or fk["match_pct"] >= 50:
                by_name[ct].setdefault("foreign_keys", []).append(fk)
        except Exception as e:  # noqa: BLE001
            by_name[ct].setdefault("foreign_key_errors", []).append({"column": cc, "error": safe_err(e)})


def family_of(simple_type):
    s = simple_type.lower()
    if re.match(r"(tinyint|smallint|int|bigint|decimal|double|float)", s):
        return "number"
    return s.split("(")[0]


ALLOWED_PLACEHOLDERS = set(PLACEHOLDER_STRINGS) | set(PLACEHOLDER_NUMBERS) | set(PLACEHOLDER_DATES)


def safety_check(profile, req):
    problems = []

    def shape_ok(s):
        return all((not ch.isalpha() or ch in "Aa") and (not ch.isdigit() or ch == "9") for ch in s)

    for t in profile["tables"]:
        for c in t["columns"]:
            where = f"{t['name']}.{c['name']}"
            cat = c.get("categorical")
            if cat:
                if cat.get("_pending"):
                    problems.append(f"{where}: labels not assigned")
                elif cat["labels"] == "masked" and not all(re.fullmatch(r"v\d{2,}", v["label"]) for v in cat["values"]):
                    problems.append(f"{where}: unmasked label")
                elif cat["labels"] == "real" and where.lower() not in req["_allowed"]:
                    problems.append(f"{where}: real labels without owner approval")
                elif cat["labels"] == "boolean" and not all(v["label"] in ("true", "false") for v in cat["values"]):
                    problems.append(f"{where}: unexpected boolean label")
            for block in ("text_shape", "id_pattern"):
                for s in (c.get(block) or {}).get("top_shapes", []):
                    if not shape_ok(s["shape"]):
                        problems.append(f"{where}: shape not masked")
            for p in c.get("placeholders", []):
                if p["value"] not in ALLOWED_PLACEHOLDERS:
                    problems.append(f"{where}: placeholder not on the allow list")
            if c.get("pii") is not None:
                problems.append(f"{where}: pii must be left for the owner")
        for s in ((t["source"].get("file") or {}).get("file_name_shapes") or []):
            if not shape_ok(s["shape"]):
                problems.append(f"{t['name']}: file name shape not masked")
    return problems


PRIVATE_KEYS = {"_pending", "_nonnull"}


def strip_private(obj):
    if isinstance(obj, dict):
        return {k: strip_private(v) for k, v in obj.items() if k not in PRIVATE_KEYS}
    if isinstance(obj, list):
        return [strip_private(v) for v in obj]
    return obj


def run_profiler():
    req = load_request()
    results, loaders, skipped, errors = [], {}, [], []

    for name in lakehouse_tables(req):
        print(f"table: {name}")
        try:
            src, change = delta_metadata(name)
            loaders[name] = (lambda nm=name: spark.table(q(nm)))
            t, raw = profile_dataframe(name, loaders[name](), "delta", req, {"source": src})
            t.setdefault("change_behaviour", {}).update(change)
            results.append((t, raw))
        except Exception as e:  # noqa: BLE001
            errors.append({"name": name, "error": safe_err(e)})

    for rel in file_folders(req):
        spec = next((d for d in req["file_datasets"] if d["path"] == rel), {})
        dname = f"files:{rel}"
        print(f"files: {rel}")
        try:
            info = inspect_folder(rel)
            if info["format"] == "delta":
                loaders[dname] = (lambda r=rel: spark.read.format("delta").load(spark_files_path(r)))
                t, raw = profile_dataframe(dname, loaders[dname](), "delta", req)
            elif not info["files"] or not (spec.get("format") or info["format"]):
                skipped.append({"name": dname, "reason": "no readable files (unknown format or empty)"})
                continue
            else:
                df, extra = read_file_dataset(rel, spec, info)
                loaders[dname] = (lambda d=df: d)
                t, raw = profile_dataframe(dname, df, "files", req, extra)
                if info.get("mixed"):
                    t["source"]["file"]["note"] = "folder holds more than one file type; most common type was read"
            results.append((t, raw))
        except Exception as e:  # noqa: BLE001
            errors.append({"name": dname, "error": safe_err(e)})

    for wname in req["warehouse_tables"]:
        dname = f"warehouse:{wname}"
        print(f"warehouse: {wname}")
        try:
            df = read_warehouse_table(wname)
            loaders[dname] = (lambda d=df: d)
            t, raw = profile_dataframe(dname, df, "warehouse", req,
                                       {"source": {"note": "types shown are Spark types read through the connector"}})
            results.append((t, raw))
        except Exception as e:  # noqa: BLE001
            errors.append({"name": dname, "error": safe_err(e)})

    assign_labels(results, req)
    infer_foreign_keys(results, loaders, req)

    profile = {
        "profile_version": 1,
        "profiler_version": PROFILER_VERSION,
        "profiled_at": dt.date.today().isoformat(),
        "spark_version": spark.version,
        "tables": [t for t, _ in results],
        "skipped": skipped,
        "errors": errors,
        "notes_for_owner": [
            "Mark PII columns yourself (pii is always null here).",
            "Add business rules the profiler cannot see, e.g. derived columns (total = qty * price).",
            "Check primary_key and foreign_keys; composite keys must be added by hand.",
            "Fill description and free_text.language where useful.",
        ],
    }
    problems = safety_check(profile, req)
    if problems:
        print(f"SAFETY CHECK FAILED — no file written. {len(problems)} problem(s):")
        for p in problems[:50]:
            print(" -", p)
        raise RuntimeError("Safety check failed; output blocked.")

    out_dir = os_path(OUTPUT_DIR, "output")
    os.makedirs(out_dir, exist_ok=True)
    fname = f"schema-profile-{dt.datetime.now(dt.timezone.utc).strftime('%Y%m%d-%H%M%S')}.json"
    with open(os.path.join(out_dir, fname), "w", encoding="utf-8") as fh:
        json.dump(strip_private(profile), fh, indent=2, default=str)
    print(f"\nDone. {len(results)} dataset(s) profiled, {len(skipped)} skipped, {len(errors)} error(s).")
    print(f"Download: Files/{OUTPUT_DIR}/output/{fname}")
    return os.path.join(out_dir, fname)

In [ ]:
run_profiler()